# Simple BM4 inspection

Run `BM4Implicit` on one guiding-centre particle in a small reproducible periodic potential. The short integration is intended to inspect states and nonlinear/projection diagnostics, not to compare accuracy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from contracts.problem import InitialValueProblem
from contracts.request import SimulationRequest
from dynamics import GuidingCenterDynamics
from initial_conditions import GCInitialConfiguration
from methods.extended.bm4 import BM4Implicit
from potential import Grid, Potential
from simulation.runner import simulate

## Define a small problem

The random potential uses a fixed seed. The state is one particle at the cell centre plus a small x offset; coordinates are packed as `(x, y)`. BM4 performs one reduced projection over each complete composition step.

In [ ]:
grid = Grid.periodic(32, 32, period=2 * np.pi)
potential = Potential.random(A=0.05, M=3, nx=grid.nx, ny=grid.ny, seed=27)
dynamics = GuidingCenterDynamics(potential, rho=0.3)
initial = GCInitialConfiguration.from_components(x=np.array([np.pi + 0.1]), y=np.array([np.pi]))
problem = InitialValueProblem(dynamics, initial)
request = SimulationRequest.uniform(t_span=(0.0, 2.0), max_step=0.1)

method = BM4Implicit(
    coupling_frequency=np.pi / 8,
    newton_absolute_tolerance=1e-12,
    newton_relative_tolerance=1e-11,
    newton_max_iterations=40,
)

In [ ]:
solution = simulate(problem, method, request)
print(f'Saved states: {solution.states.shape[1]}')
print(f'State array shape (coordinates, samples): {solution.states.shape}')
print('Diagnostic fields:', sorted(solution.diagnostics))

plt.plot(solution.states[0], solution.states[1], marker='.', ms=3)
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.title('BM4Implicit trajectory')
plt.show()

## Inspect step diagnostics

The available diagnostic names are printed above because the solution stores only fields emitted by the method. The following cell summarizes the nonlinear work and projection multiplier when those fields are present.

In [ ]:
for name in ('nonlinear_iterations', 'residual_evaluations', 'projection_multiplier_norms'):
    values = solution.diagnostics.get(name)
    if values is not None:
        array = np.asarray(values, dtype=float)
        print(f'{name}: shape={array.shape}, min={array.min():.3e}, mean={array.mean():.3e}, max={array.max():.3e}')